# 011 · Train a "player" detector on SportsMOT (Google Colab)

**hoop-reid-tracker**, experiment 011. This notebook fine-tunes YOLO26 so that it detects **players only**, not every person.

### Why
Our detector is YOLO26m trained on COCO: its class 0 is *person*, so it fires on players, **referees, coaches, bench and crowd**. SportsMOT annotates only the 10 players on court, so everything else counts as a false positive. Until now we removed those people *after* detection with hand-made rules (court mask 003/C06, referee classifier 006).

The best published trackers on SportsMOT (Deep-EIoU, MixSort, DiffMOT) do the opposite. They **fine-tune the detector on SportsMOT**, so it learns the annotation policy from the whole image: jersey, position, pose and context, not a colour rule. Deep-EIoU reaches MOTA ≈ 96 with no court mask at all.

### What the model learns
* **Positives**: boxes of the 10 players.
* **Negatives**: every other pixel, *including the referees and the bench*. Nobody labels them "referee"; they are simply never labelled "player". In COCO they were positives (*person*); here the model is punished each time it fires on them. That is how the referee disappears from the detections.
* Frames with no player (close-ups, crowd shots) have an empty label file. They are pure negatives, and useful.

### Splits (no leakage)
| split | content | role |
|---|---|---|
| `train` | 12 SportsMOT **train** clips (3 games), 1 frame in 3 | learn |
| `val` | 3 held-out train clips, one per game | choose the best epoch (same arenas, so it measures *fitting*) |
| `test` | the 15 SportsMOT **val** clips (4 other games), 1 frame in 10 | **final check on arenas never seen**: looked at only after training |

Our tracking results are measured on the SportsMOT val clips, so they must never influence training. That is why they are only the *test* split here.

### Before running (on your laptop)
1. `python scripts/build_player_dataset.py` creates `data/player_det.zip`.
2. Upload `player_det.zip` to Google Drive, in a folder `MyDrive/hooptrack/`.
3. In Colab: **Runtime → Change runtime type → T4 GPU**, then run the cells in order.

**Time on a free T4:** about 3–5 min per epoch at 1280 px, so ~2–3 h for 40 epochs. Weights are written to Drive **every epoch**. If Colab disconnects, run sections 1–3 again and then section 5: training **resumes** from the last epoch.

## 1 · GPU and libraries
`nvidia-smi` must show a GPU (T4, L4 or A100). If it says *command not found*, change the runtime type.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "ultralytics>=8.4"   # YOLO26 needs ultralytics 8.4 or newer
import ultralytics
ultralytics.checks()

## 2 · Configuration
Everything you may want to change is in this cell.

* `MODEL`: start from COCO weights. The network already knows what a person looks like; we only teach it *which* people count (transfer learning). `yolo26m` is the same size as our current detector, so the comparison is fair and the CPU speed on the laptop is unchanged.
* `IMGSZ = 1280`: the same resolution as our pipeline. Players are small in 1280×720 broadcasts, and a smaller input loses the far ones.
* `BATCH`: images per step. 4 fits a T4 at 1280 px (8 runs out of memory); try 8 on an L4/A100.
* `HOURS`: optional time budget. Ultralytics stops at that time and stretches the learning-rate schedule to fit.

In [ ]:
import os
from pathlib import Path

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/hooptrack")   # where you uploaded player_det.zip
    WORK = Path("/content")                                # fast local disk of the Colab machine
else:                                                       # running elsewhere (e.g. a local GPU)
    DRIVE_DIR = Path(os.environ.get("HOOPTRACK_DIR", "data"))
    WORK = Path(os.environ.get("HOOPTRACK_WORK", "/tmp/hooptrack"))

ZIP = DRIVE_DIR / "player_det.zip"
RUNS = DRIVE_DIR / "runs"            # checkpoints on Drive: they survive a disconnection
NAME = "player_yolo26m_1280"         # run name (one folder per run)

MODEL = os.environ.get("HOOPTRACK_MODEL", "yolo26m.pt")
IMGSZ = int(os.environ.get("HOOPTRACK_IMGSZ", 1280))
EPOCHS = int(os.environ.get("HOOPTRACK_EPOCHS", 40))
PATIENCE = 10                        # stop if val mAP does not improve for 10 epochs
BATCH = int(os.environ.get("HOOPTRACK_BATCH", 4))   # yolo26m @1280 does not fit batch 8 on a T4 (15 GB)
HOURS = None                         # e.g. 2.5 to fit a session; None = no time limit
WORKERS = 2                          # Colab has 2 CPU cores

assert ZIP.is_file(), f"{ZIP} not found: upload player_det.zip to MyDrive/hooptrack/"
RUNS.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
print("zip:", ZIP, f"({ZIP.stat().st_size / 1e6:.0f} MB) | runs:", RUNS)

## 3 · Unpack the dataset on the local disk
Reading thousands of small files directly from Drive is very slow. One copy of the zip followed by unpacking takes ~1 minute and makes every epoch faster.

`data.yaml` is written with absolute paths for this machine.

In [ ]:
import shutil, zipfile, yaml

DATA_ROOT = WORK / "player_det"
if not (DATA_ROOT / "images" / "train").is_dir():
    local_zip = WORK / "player_det.zip"
    shutil.copy(ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(WORK)
    local_zip.unlink()

cfg = yaml.safe_load((DATA_ROOT / "data.yaml").read_text())
cfg["path"] = str(DATA_ROOT)
DATA = DATA_ROOT / "data_colab.yaml"
DATA.write_text(yaml.safe_dump(cfg, sort_keys=False))

for split in ("train", "val", "test"):
    labels = list((DATA_ROOT / "labels" / split).glob("*.txt"))
    boxes = sum(len(p.read_text().split("\n")) for p in labels if p.stat().st_size)
    empty = sum(1 for p in labels if not p.stat().st_size)
    clips = len({p.stem.rsplit("_", 1)[0] for p in labels})
    print(f"{split:5s}: {len(labels):5d} images, {clips:2d} clips, {boxes:6d} player boxes, {empty} images without players")

### Look at the labels
Always check the labels before training. Green boxes are what the model will learn as "player". Notice that the referees and the bench have **no box**: for the model they are background.

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt
import numpy as np

def read_labels(img_path: Path) -> np.ndarray:
    # YOLO label (class cx cy w h, normalised) -> pixel boxes x1 y1 x2 y2
    lbl = Path(str(img_path).replace("/images/", "/labels/")).with_suffix(".txt")
    rows = [list(map(float, l.split()[1:])) for l in lbl.read_text().splitlines() if l.strip()]
    if not rows:
        return np.zeros((0, 4))
    img = cv2.imread(str(img_path)); h, w = img.shape[:2]
    r = np.array(rows)
    return np.c_[(r[:, 0] - r[:, 2] / 2) * w, (r[:, 1] - r[:, 3] / 2) * h,
                 (r[:, 0] + r[:, 2] / 2) * w, (r[:, 1] + r[:, 3] / 2) * h]

random.seed(0)
samples = random.sample(sorted((DATA_ROOT / "images" / "train").glob("*.jpg")), 4)
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, p in zip(axes.ravel(), samples):
    img = cv2.imread(str(p))
    for x1, y1, x2, y2 in read_labels(p).astype(int):
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    ax.imshow(img[..., ::-1]); ax.set_title(p.stem, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 4 · Baseline: the COCO detector we use today
Before training, measure the current detector on the same images, so we have a number to beat.

For the COCO model we keep only class 0 (*person*) and compare it with the *player* boxes. Every referee or bench member it detects counts as a **false positive**. That lowers its **precision** (P = correct detections / all detections) and its mAP.

* **mAP50**: average precision when a detection counts as correct if it overlaps a real player by IoU ≥ 0.5.
* **mAP50-95**: the same, averaged over stricter overlaps (IoU 0.5 to 0.95). It rewards tight boxes.
* **R** (recall): the fraction of real players found.

In [ ]:
from ultralytics import YOLO

def evaluate(weights, split: str, coco: bool = False) -> dict:
    model = YOLO(weights)
    m = model.val(data=str(DATA), split=split, imgsz=IMGSZ, batch=BATCH, conf=0.001, iou=0.7,
                  classes=[0] if coco else None, plots=False, verbose=False,
                  project=str(WORK / "val_runs"), name=f"{Path(str(weights)).stem}_{split}", exist_ok=True)
    return {"P": round(m.box.mp, 3), "R": round(m.box.mr, 3),
            "mAP50": round(m.box.map50, 3), "mAP50-95": round(m.box.map, 3)}

BASE = f"COCO {Path(MODEL).stem} (person)"   # label of the baseline in the tables
results = {}
for split in ("val", "test"):
    results[(BASE, split)] = evaluate(MODEL, split, coco=True)
    print(split, results[(BASE, split)])

## 5 · Training
The settings, and why:

| setting | value | why |
|---|---|---|
| `single_cls` | True | one class, *player* (index 0, like COCO *person*, so the tracker code does not change) |
| `hsv_h / hsv_s / hsv_v` | 0.3 / 0.7 / 0.4 | strong colour changes. Train has only 3 arenas, and the model must not memorise *their* floor and jersey colours (the lesson of C06) |
| `scale`, `translate` | 0.5, 0.1 | zoomed-in and zoomed-out shots |
| `fliplr` | 0.5 | a mirrored court is still a court |
| `mosaic`, `close_mosaic` | 1.0, 10 | mosaic (4 images in one) adds variety; it is switched off for the last 10 epochs so the model ends on real-looking frames |
| `patience` | 10 | early stop when val mAP stops improving |

**If Colab disconnects:** run sections 1–3 again and then this cell. It finds `last.pt` on Drive and resumes (same epoch, same optimiser state).

In [ ]:
import torch

last = RUNS / NAME / "weights" / "last.pt"
# A finished run stores epoch = -1 in last.pt (the optimizer state is stripped): nothing to resume.
finished = last.is_file() and torch.load(last, map_location="cpu", weights_only=False).get("epoch", -1) == -1
if finished:
    print("Training already finished:", last.parent, "(delete or rename the run folder to train again)")
elif last.is_file():
    print("Resuming from", last)
    model = YOLO(str(last))
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    model.train(
        data=str(DATA), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, workers=WORKERS,
        time=HOURS, project=str(RUNS), name=NAME, exist_ok=True, seed=0, plots=True,
        single_cls=True,
        hsv_h=0.3, hsv_s=0.7, hsv_v=0.4, scale=0.5, translate=0.1, fliplr=0.5, degrees=0.0,
        mosaic=1.0, close_mosaic=10,
    )
BEST = RUNS / NAME / "weights" / "best.pt"
print("best weights:", BEST)

### Training curves
`results.png` shows the losses (they should go down) and the val metrics (they should go up, then flatten). If val mAP falls while the train loss keeps falling, the model is **overfitting** the 3 training arenas.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(RUNS / NAME / "results.png"), width=1100))

## 6 · Evaluation: before vs after
The row that matters is **test**: 4 arenas the model has never seen. If the fine-tuned model wins there, it generalises. If it wins only on val, it has learned the training arenas, and we need more varied data.

In [ ]:
import pandas as pd

for split in ("val", "test"):
    results[("fine-tuned player", split)] = evaluate(BEST, split)
table = pd.DataFrame(results).T
table.index.names = ["model", "split"]
display(table.sort_index(level=1))

### What the tracker will see: errors per frame at its threshold
mAP sweeps every confidence threshold. The tracker, however, treats boxes with score ≥ 0.5 as *sure* (`high_thresh`). On the test frames, at that threshold, we count per frame:
* **FP**: detections that are not a player (referee, bench, crowd, duplicate). Each one can become a false identity.
* **FN**: players not detected. Each one is a hole in a track.

In [ ]:
def iou(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def match(pred: np.ndarray, gt: np.ndarray, thr: float = 0.5):
    # greedy one-to-one matching, highest IoU first -> (matched pred mask, matched gt mask)
    mp, mg = np.zeros(len(pred), bool), np.zeros(len(gt), bool)
    if len(pred) and len(gt):
        m = iou(pred, gt)
        for k in np.argsort(-m, axis=None):
            i, j = np.unravel_index(k, m.shape)
            if m[i, j] < thr:
                break
            if not mp[i] and not mg[j]:
                mp[i] = mg[j] = True
    return mp, mg

test_imgs = sorted((DATA_ROOT / "images" / "test").glob("*.jpg"))
models = {BASE: (YOLO(MODEL), [0]), "fine-tuned": (YOLO(str(BEST)), None)}
counts = {k: {"TP": 0, "FP": 0, "FN": 0} for k in models}
per_image = {k: {} for k in models}
for p in test_imgs:
    gt = read_labels(p)
    for name, (m, cls) in models.items():
        r = m.predict(str(p), imgsz=IMGSZ, conf=0.5, classes=cls, verbose=False)[0]
        pred = r.boxes.xyxy.cpu().numpy()
        mp, mg = match(pred, gt)
        counts[name]["TP"] += int(mp.sum()); counts[name]["FP"] += int((~mp).sum()); counts[name]["FN"] += int((~mg).sum())
        per_image[name][p] = (pred, mp)

n = len(test_imgs)
summary = pd.DataFrame({k: {"FP per frame": v["FP"] / n, "FN per frame": v["FN"] / n,
                            "precision": v["TP"] / max(v["TP"] + v["FP"], 1),
                            "recall": v["TP"] / max(v["TP"] + v["FN"], 1)} for k, v in counts.items()}).T.round(3)
display(summary)

### Side by side
Left: COCO. Right: fine-tuned. **Blue** = correct detection, **red** = false positive, **green** = real player (thin). The test images shown are the ones where the two models differ most in false positives.

In [ ]:
diff = sorted(test_imgs, key=lambda p: -abs(int((~per_image[BASE][p][1]).sum())
                                             - int((~per_image["fine-tuned"][p][1]).sum())))[:4]
fig, axes = plt.subplots(len(diff), 2, figsize=(18, 5 * len(diff)))
for row, p in zip(np.atleast_2d(axes), diff):
    gt = read_labels(p)
    for ax, name in zip(row, models):
        img = cv2.imread(str(p))
        for x1, y1, x2, y2 in gt.astype(int):
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 1)
        pred, mp = per_image[name][p]
        for (x1, y1, x2, y2), ok in zip(pred.astype(int), mp):
            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 120, 0) if ok else (0, 0, 255), 2)
        ax.imshow(img[..., ::-1]); ax.axis("off")
        ax.set_title(f"{name} | {p.stem} | FP {int((~mp).sum())}", fontsize=10)
plt.tight_layout()
plt.savefig(RUNS / NAME / "test_side_by_side.jpg", dpi=80)
plt.show()

## 7 · Save and take it home
The weights and the numbers are written next to the run on Drive. Then:

1. Download `MyDrive/hooptrack/runs/player_yolo26m_1280/weights/best.pt` and save it in the repo as `weights/yolo26m_player.pt` (the `weights/` folder is git-ignored).
2. Measure it **in the tracker** with the same script and clips as 010, changing only the detector:

```powershell
python scripts/run_court_source_eval.py --set detector.weights=weights/yolo26m_player.pt `
    --cache-dir outputs/detections/yolo26m_player --out outputs/runs/011_player_detector `
    --variants colour segmentation `
    --clips v_00HRwkvvjtQ_c005 v_5ekaksddqrc_c002 v_BgwzTUxJaeU_c012 v_4r8QL_wglzQ_c001
```

Then compare with `outputs/runs/010_court_source/summary.md`.

In [ ]:
import json
report = {
    "model": MODEL, "imgsz": IMGSZ, "epochs_max": EPOCHS, "batch": BATCH,
    "map_table": {f"{m} | {s}": v for (m, s), v in results.items()},
    "test_at_conf_0.5": summary.to_dict(orient="index"),
}
(RUNS / NAME / "eval_011.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
print("\nDownload:", BEST)